# Advanced RAG with LangChain: HR Chatbot

This notebook focuses on building a production-ready RAG (Retrieval-Augmented Generation) system using LangChain.

## What We'll Build
- **HR Chatbot** that answers questions from company policy documents
- Supports multiple file formats (PDF, CSV, TXT, TSV)
- Advanced retrieval techniques (MMR, Multi-Query, Compression)
- Conversational RAG with chat history
- Interactive Gradio UI

---

## 1. Setup & Installation

In [ ]:
# Install dependencies
!pip install -q \
    langchain>=0.3.0 \
    langchain-openai>=0.2.0 \
    langchain-community>=0.3.0 \
    langchain-core>=0.3.0 \
    chromadb>=0.5.0 \
    pypdf>=4.0 \
    python-dotenv>=1.0 \
    pandas>=2.0 \
    sentence-transformers>=3.0 \
    gradio>=4.0 \
    fpdf2>=2.7.0

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# ============================================================
# CONFIGURATION: Choose OpenRouter OR OpenAI
# ============================================================
USE_OPENROUTER = True  # Set to False for direct OpenAI

if USE_OPENROUTER:
    API_KEY = os.getenv("OPENROUTER_API_KEY")
    BASE_URL = "https://openrouter.ai/api/v1"
    MODEL_NAME = "openai/gpt-4o-mini"
    print("Using OpenRouter API")
else:
    API_KEY = os.getenv("OPENAI_API_KEY")
    BASE_URL = None
    MODEL_NAME = "gpt-4o-mini"
    print("Using OpenAI API")

print("Configuration loaded!")

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize LLM
llm = ChatOpenAI(
    model=MODEL_NAME,
    openai_api_key=API_KEY,
    openai_api_base=BASE_URL,
    temperature=0.7,
)

# Initialize Embeddings (free, runs locally)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# Test
print("LLM:", llm.invoke("Say 'RAG is ready!'").content)
print("Embeddings dimension:", len(embeddings.embed_query("test")))

---
## 2. Create HR Documents (PDF)

We'll create realistic HR policy documents to use as our knowledge base.

In [ ]:
from fpdf import FPDF
import os

os.makedirs("hr_documents", exist_ok=True)

# HR Document 1: Employee Handbook
employee_handbook = """
EMPLOYEE HANDBOOK
TechCorp Inc. | Last Updated: January 2024

CHAPTER 1: WELCOME
Welcome to TechCorp Inc.! This handbook outlines our company policies, procedures, and benefits.

CHAPTER 2: WORKING HOURS AND ATTENDANCE

Standard Working Hours:
- Regular hours: Monday to Friday, 9:00 AM to 6:00 PM
- Core hours (mandatory): 10:00 AM to 4:00 PM
- Flexible start: 8:00 AM to 10:00 AM
- Flexible end: 5:00 PM to 7:00 PM

Remote Work Policy:
- Up to 3 days remote per week after 90-day probation
- Submit requests 24 hours in advance
- Must be available during core hours
- High-speed internet required

Attendance:
- Log attendance through HR portal daily
- Notify manager if late by more than 15 minutes
- Three unexcused absences may result in disciplinary action

CHAPTER 3: LEAVE POLICIES

Annual Leave:
- 20 days paid annual leave per year
- Accrues at 1.67 days per month
- Maximum carry-forward: 5 days
- Submit requests 2 weeks in advance for periods over 3 days

Sick Leave:
- 12 days paid sick leave per year
- Medical certificate required for absences over 2 days
- Does not carry forward

Parental Leave:
- Maternity Leave: 16 weeks fully paid
- Paternity Leave: 4 weeks fully paid
- Adoption Leave: 12 weeks fully paid
- Additional unpaid leave up to 6 months available

Other Leave:
- Bereavement: 5 days (immediate family), 3 days (extended)
- Marriage: 5 days
- Jury Duty: Fully paid
- Voting: 2 hours on election day

CHAPTER 4: COMPENSATION AND BENEFITS

Salary:
- Paid on last working day of each month
- Direct deposit mandatory
- Annual reviews in March
- Performance bonuses paid in December

Health Insurance:
- Comprehensive coverage for employee and dependents
- Includes medical, dental, and vision
- Company pays 80%, employee pays 20%
- Coverage begins on first day

Retirement Benefits:
- 401(k) with company match up to 6%
- Vesting: 25% per year over 4 years

Other Benefits:
- Life insurance: 2x annual salary
- Disability insurance (short and long-term)
- Employee Assistance Program (EAP)
- Gym reimbursement: Up to $50/month
- Professional development: $2,000/year
- Stock options (manager level and above)
"""

# HR Document 2: Code of Conduct
code_of_conduct = """
CODE OF CONDUCT
TechCorp Inc.

INTRODUCTION
This Code outlines principles and standards guiding behavior at TechCorp.

PROFESSIONAL BEHAVIOR

Respect and Dignity:
- Treat all colleagues with respect and dignity
- Discrimination based on race, gender, age, religion, disability, or sexual orientation is prohibited
- Harassment of any kind will not be tolerated
- Report violations to HR or use anonymous ethics hotline

Workplace Safety:
- Follow all safety guidelines
- Report unsafe conditions immediately
- No weapons on premises
- Drug and alcohol use during work hours prohibited

CONFLICTS OF INTEREST
- Disclose potential conflicts to manager and HR
- Do not accept gifts over $100 from vendors
- No outside employment conflicting with duties
- Do not use company resources for personal gain

CONFIDENTIALITY
- Do not disclose confidential information
- All employees must sign NDA
- Handle customer data per privacy policy
- Breach may result in immediate termination

COMMUNICATION

Internal:
- Use professional language
- Respond to emails within 24 hours
- Slack for quick queries, email for formal communication

External:
- Only authorized spokespersons may speak to media
- Social media posts must not disclose confidential information
- Company logo use requires Marketing approval

DISCIPLINARY PROCESS
1. Verbal Warning
2. Written Warning
3. Performance Improvement Plan
4. Suspension
5. Termination

REPORTING VIOLATIONS
- Report to supervisor or HR
- Anonymous hotline: 1-800-ETHICS-1
- Email: ethics@techcorp.com
"""

# HR Document 3: Performance Policy
performance_policy = """
PERFORMANCE MANAGEMENT POLICY
TechCorp Inc.

OVERVIEW
TechCorp fosters employee growth through regular feedback and fair evaluations.

REVIEW CYCLE

Annual Reviews:
- Conducted in March
- Covers January-December performance
- Self-assessment required
- Calibration meetings ensure fairness

Mid-Year Check-ins:
- Conducted in August/September
- Focus on goal progress
- Not tied to compensation

RATING SCALE
1 - Needs Improvement: Below expectations
2 - Partially Meets: Occasionally meets expectations
3 - Meets Expectations: Consistently meets requirements
4 - Exceeds Expectations: Frequently surpasses requirements
5 - Outstanding: Exceptional, role model

GOAL SETTING (SMART)
- Specific: Clear and defined
- Measurable: Quantifiable criteria
- Achievable: Realistic
- Relevant: Aligned with objectives
- Time-bound: Clear deadline

MERIT INCREASES
- Rating 1-2: 0% increase
- Rating 3: 2-4% increase
- Rating 4: 4-6% increase
- Rating 5: 6-10% increase

PROMOTIONS
- Require consistent Exceeds or Outstanding ratings
- Manager recommendation required
- Skip-level approval needed

PERFORMANCE IMPROVEMENT PLAN (PIP)

When Required:
- Two consecutive Needs Improvement ratings
- Serious performance issues

Components:
- Clear statement of gaps
- Specific improvement actions
- 60-90 day duration
- Weekly check-ins

Outcomes:
- Success: Return to normal management
- Failure: May result in termination
"""

# HR Document 4: IT Security Policy
it_security_policy = """
IT AND SECURITY POLICY
TechCorp Inc.

PURPOSE
Guidelines for secure use of IT resources.

ACCEPTABLE USE

Permitted:
- Business activities
- Limited personal use during breaks
- Professional development

Prohibited:
- Inappropriate or illegal content
- Unauthorized software installation
- Sharing login credentials
- Personal business via company email
- Cryptocurrency mining
- Torrenting/pirating

PASSWORDS
- Minimum 12 characters
- Include uppercase, lowercase, numbers, symbols
- Change every 90 days
- No reuse of last 10 passwords
- Use LastPass (provided free)

MULTI-FACTOR AUTHENTICATION
- Required for all systems
- Use authenticator app (not SMS)
- Report lost devices immediately

DATA CLASSIFICATION
- Public: Can share externally
- Internal: Employees only
- Confidential: Need-to-know basis
- Restricted: Special access required

DATA HANDLING
- Never store confidential data on personal devices
- Use encrypted drives
- Shred sensitive physical documents
- Report breaches within 24 hours

DEVICES

Company Devices:
- Company property, return upon termination
- Keep locked when unattended
- Enable full-disk encryption

BYOD:
- Register with IT first
- MDM software required
- Company can remotely wipe data

REMOTE ACCESS
- Use company VPN for all remote work
- Never use public WiFi without VPN
- Secure home WiFi with strong password
- Lock screen when stepping away

INCIDENT REPORTING
Report immediately:
- Phishing emails
- Lost/stolen devices
- Malware detection
- Unauthorized access attempts

Contact: security@techcorp.com or Extension 5555
"""

def create_pdf(content: str, filename: str, title: str):
    """Create a PDF from text content."""
    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)
    
    # Title
    pdf.set_font("Helvetica", "B", 16)
    pdf.cell(0, 10, title, ln=True, align="C")
    pdf.ln(10)
    
    # Content
    pdf.set_font("Helvetica", "", 10)
    
    for line in content.strip().split('\n'):
        if line.strip().isupper() or line.strip().startswith('CHAPTER'):
            pdf.set_font("Helvetica", "B", 12)
            pdf.ln(5)
            pdf.multi_cell(0, 6, line.strip())
            pdf.set_font("Helvetica", "", 10)
        elif line.strip().startswith('-'):
            pdf.multi_cell(0, 5, "  " + line.strip())
        elif line.strip():
            pdf.multi_cell(0, 5, line.strip())
        else:
            pdf.ln(3)
    
    pdf.output(filename)
    print(f"Created: {filename}")

# Create all PDFs
create_pdf(employee_handbook, "hr_documents/employee_handbook.pdf", "Employee Handbook")
create_pdf(code_of_conduct, "hr_documents/code_of_conduct.pdf", "Code of Conduct")
create_pdf(performance_policy, "hr_documents/performance_policy.pdf", "Performance Policy")
create_pdf(it_security_policy, "hr_documents/it_security_policy.pdf", "IT Security Policy")

print("\nAll HR documents created!")

---
## 3. Document Loading

Load documents from multiple file formats.

In [ ]:
from langchain_community.document_loaders import (
    PyPDFLoader,
    DirectoryLoader,
    CSVLoader,
    TextLoader,
)
from langchain.schema import Document
import pandas as pd

def load_documents(directory: str = "hr_documents") -> list[Document]:
    """
    Load all documents from a directory.
    Supports: PDF, CSV, TXT, TSV
    """
    all_docs = []
    
    # Load PDFs
    pdf_loader = DirectoryLoader(
        directory,
        glob="**/*.pdf",
        loader_cls=PyPDFLoader,
        show_progress=True
    )
    try:
        pdf_docs = pdf_loader.load()
        all_docs.extend(pdf_docs)
        print(f"Loaded {len(pdf_docs)} pages from PDFs")
    except Exception as e:
        print(f"No PDFs found or error: {e}")
    
    # Load TXT files
    for txt_file in [f for f in os.listdir(directory) if f.endswith('.txt')]:
        try:
            loader = TextLoader(os.path.join(directory, txt_file))
            all_docs.extend(loader.load())
            print(f"Loaded: {txt_file}")
        except Exception as e:
            print(f"Error loading {txt_file}: {e}")
    
    # Load CSV files
    for csv_file in [f for f in os.listdir(directory) if f.endswith('.csv')]:
        try:
            loader = CSVLoader(os.path.join(directory, csv_file))
            all_docs.extend(loader.load())
            print(f"Loaded: {csv_file}")
        except Exception as e:
            print(f"Error loading {csv_file}: {e}")
    
    # Load TSV files
    for tsv_file in [f for f in os.listdir(directory) if f.endswith('.tsv')]:
        try:
            df = pd.read_csv(os.path.join(directory, tsv_file), sep='\t')
            for idx, row in df.iterrows():
                content = " | ".join([f"{col}: {val}" for col, val in row.items()])
                all_docs.append(Document(
                    page_content=content,
                    metadata={"source": tsv_file, "row": idx}
                ))
            print(f"Loaded: {tsv_file}")
        except Exception as e:
            print(f"Error loading {tsv_file}: {e}")
    
    return all_docs

# Load all documents
documents = load_documents("hr_documents")
print(f"\nTotal documents loaded: {len(documents)}")

---
## 4. Text Splitting (Chunking)

Split documents into smaller chunks for better retrieval.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Configure text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # Characters per chunk
    chunk_overlap=200,    # Overlap between chunks
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]  # Split hierarchy
)

# Split documents
chunks = text_splitter.split_documents(documents)

print(f"Original documents: {len(documents)}")
print(f"After splitting: {len(chunks)} chunks")
print(f"\nSample chunk:")
print(f"Content: {chunks[0].page_content[:200]}...")
print(f"Metadata: {chunks[0].metadata}")

---
## 5. Vector Store & Indexing

Create embeddings and store in ChromaDB.

In [ ]:
from langchain_community.vectorstores import Chroma

# Create vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./hr_chroma_db",
    collection_name="hr_policies"
)

print(f"Vector store created with {vectorstore._collection.count()} documents")

In [ ]:
# Test similarity search
query = "How many vacation days do I get?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
print("Top 3 results:")
for i, doc in enumerate(results):
    print(f"\n{i+1}. [{doc.metadata.get('source', 'Unknown')}]")
    print(f"   {doc.page_content[:150]}...")

---
## 6. Basic RAG Chain

Build a simple RAG pipeline using LCEL.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_template("""
You are an HR Assistant for TechCorp Inc. Answer questions based ONLY on the context provided.

Rules:
- Be helpful and professional
- If the answer isn't in the context, say "I don't have that information in our HR policies."
- Cite specific policies when possible
- Use bullet points for clarity

Context:
{context}

Question: {question}

Answer:
""")

# Format retrieved documents
def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('source', 'Unknown')}]\n{doc.page_content}" 
        for doc in docs
    )

# Build RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("Basic RAG chain created!")

In [ ]:
# Test basic RAG
questions = [
    "How many days of annual leave do I get?",
    "What is the remote work policy?",
    "What are the password requirements?",
    "How do promotions work?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print("-" * 50)

---
## 7. Advanced Retrieval Techniques

### 7.1 MMR (Maximum Marginal Relevance)
Balances relevance with diversity - avoids redundant results.

In [ ]:
# MMR Retriever
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 4,           # Return 4 documents
        "fetch_k": 10,    # Fetch 10, then select diverse 4
        "lambda_mult": 0.5  # 0=max diversity, 1=max relevance
    }
)

# Test MMR
query = "Tell me about employee benefits"
mmr_docs = mmr_retriever.invoke(query)

print(f"MMR Results for: '{query}'\n")
for i, doc in enumerate(mmr_docs):
    print(f"{i+1}. [{doc.metadata.get('source', 'Unknown')}]")
    print(f"   {doc.page_content[:100]}...\n")

### 7.2 Multi-Query Retriever
Generates multiple query variations for better recall.

In [ ]:
from langchain.retrievers import MultiQueryRetriever

# Multi-Query Retriever
multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=retriever,
    llm=llm
)

# Test
query = "What happens if I perform poorly?"
multi_docs = multi_query_retriever.invoke(query)

print(f"Multi-Query Results for: '{query}'")
print(f"Retrieved {len(multi_docs)} unique documents\n")
for i, doc in enumerate(multi_docs[:3]):
    print(f"{i+1}. {doc.page_content[:100]}...")

### 7.3 Contextual Compression
Extracts only the relevant parts from retrieved documents.

In [ ]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

# Create compressor
compressor = LLMChainExtractor.from_llm(llm)

# Compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=retriever
)

# Test
query = "How much maternity leave is provided?"
compressed_docs = compression_retriever.invoke(query)

print(f"Compressed Results for: '{query}'\n")
for i, doc in enumerate(compressed_docs):
    print(f"{i+1}. {doc.page_content}\n")

### 7.4 Similarity Score Threshold
Only return documents above a similarity threshold.

In [ ]:
# Threshold retriever
threshold_retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "score_threshold": 0.5,  # Only return if similarity > 0.5
        "k": 5
    }
)

# Test with relevant query
query = "What is the 401k match?"
threshold_docs = threshold_retriever.invoke(query)
print(f"Relevant query: Found {len(threshold_docs)} documents above threshold")

# Test with irrelevant query
query = "What is the best pizza in town?"
threshold_docs = threshold_retriever.invoke(query)
print(f"Irrelevant query: Found {len(threshold_docs)} documents above threshold")

---
## 8. Conversational RAG (with Chat History)

RAG that remembers previous messages in the conversation.

In [ ]:
from langchain.chains import create_history_aware_retriever, create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Prompt to contextualize questions based on chat history
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system", """Given the chat history and a follow-up question, 
    rephrase the question to be standalone. If already standalone, return as-is."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# History-aware retriever
history_aware_retriever = create_history_aware_retriever(
    llm, mmr_retriever, contextualize_prompt
)

# QA prompt
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are TechCorp's HR Assistant. Answer based on the context.
    Be concise and helpful. If unsure, say so.
    
    Context: {context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Build conversational RAG chain
qa_chain = create_stuff_documents_chain(llm, qa_prompt)
conversational_rag = create_retrieval_chain(history_aware_retriever, qa_chain)

print("Conversational RAG chain created!")

In [ ]:
# Test conversational RAG
chat_history = []

# First question
q1 = "What types of leave are available?"
result1 = conversational_rag.invoke({"input": q1, "chat_history": chat_history})
print(f"Q1: {q1}")
print(f"A1: {result1['answer']}\n")

# Update history
chat_history.extend([
    HumanMessage(content=q1),
    AIMessage(content=result1['answer'])
])

# Follow-up question (uses "it" - needs context)
q2 = "How much of it can I carry forward?"
result2 = conversational_rag.invoke({"input": q2, "chat_history": chat_history})
print(f"Q2: {q2}")
print(f"A2: {result2['answer']}\n")

# Update history
chat_history.extend([
    HumanMessage(content=q2),
    AIMessage(content=result2['answer'])
])

# Another follow-up
q3 = "What about sick leave?"
result3 = conversational_rag.invoke({"input": q3, "chat_history": chat_history})
print(f"Q3: {q3}")
print(f"A3: {result3['answer']}")

---
## 9. Complete HR Chatbot Class

Production-ready chatbot combining all techniques.

In [ ]:
class HRChatbot:
    """
    Production-ready HR RAG Chatbot.
    
    Features:
    - Multi-format document loading (PDF, CSV, TXT, TSV)
    - Advanced retrieval (MMR)
    - Conversation memory
    - Source attribution
    """
    
    def __init__(self, llm, embeddings, docs_path: str = "hr_documents"):
        self.llm = llm
        self.embeddings = embeddings
        self.docs_path = docs_path
        self.vectorstore = None
        self.chain = None
        self.chat_history = []
        
    def initialize(self):
        """Load documents and build the RAG chain."""
        print("Initializing HR Chatbot...")
        
        # Load documents
        documents = load_documents(self.docs_path)
        
        # Split into chunks
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000, 
            chunk_overlap=200
        )
        chunks = splitter.split_documents(documents)
        
        # Create vector store
        self.vectorstore = Chroma.from_documents(
            documents=chunks,
            embedding=self.embeddings,
            collection_name="hr_chatbot"
        )
        
        # Create retriever with MMR
        retriever = self.vectorstore.as_retriever(
            search_type="mmr",
            search_kwargs={"k": 5, "fetch_k": 10}
        )
        
        # Contextualize prompt
        contextualize_prompt = ChatPromptTemplate.from_messages([
            ("system", "Rephrase the question to be standalone based on chat history."),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}")
        ])
        
        # History-aware retriever
        history_retriever = create_history_aware_retriever(
            self.llm, retriever, contextualize_prompt
        )
        
        # QA prompt
        qa_prompt = ChatPromptTemplate.from_messages([
            ("system", """You are TechCorp's friendly HR Assistant.
            
Rules:
- Answer based ONLY on the provided context
- Be helpful, professional, and concise
- Use bullet points for lists
- If unsure, say "I don't have that information. Please contact HR at hr@techcorp.com"

Context: {context}"""),
            MessagesPlaceholder(variable_name="chat_history"),
            ("human", "{input}")
        ])
        
        # Build chain
        qa_chain = create_stuff_documents_chain(self.llm, qa_prompt)
        self.chain = create_retrieval_chain(history_retriever, qa_chain)
        
        print(f"Chatbot ready! Indexed {len(chunks)} chunks.")
        return self
    
    def chat(self, message: str) -> str:
        """Send a message and get a response."""
        if not self.chain:
            return "Error: Chatbot not initialized."
        
        result = self.chain.invoke({
            "input": message,
            "chat_history": self.chat_history
        })
        
        # Update history
        self.chat_history.extend([
            HumanMessage(content=message),
            AIMessage(content=result['answer'])
        ])
        
        # Keep last 20 messages
        if len(self.chat_history) > 20:
            self.chat_history = self.chat_history[-20:]
        
        return result['answer']
    
    def clear_history(self):
        """Clear conversation history."""
        self.chat_history = []
        return "History cleared!"


# Initialize chatbot
hr_bot = HRChatbot(llm, embeddings, "hr_documents")
hr_bot.initialize()

In [ ]:
# Test the chatbot
print("=== HR Chatbot Test ===")

test_questions = [
    "Hi! How many vacation days do I get?",
    "Can I work from home?",
    "What happens if I get a bad performance review?",
    "How do I report a security issue?",
]

for q in test_questions:
    print(f"\nYou: {q}")
    print(f"HR Bot: {hr_bot.chat(q)}")
    print("-" * 40)

---
## 10. Interactive Gradio UI

Launch a beautiful chat interface!

In [ ]:
import gradio as gr

# Reset history
hr_bot.clear_history()

# Create Gradio interface
with gr.Blocks(
    title="HR Assistant",
    theme=gr.themes.Soft(primary_hue="blue"),
    css="""
    .gradio-container { max-width: 800px !important; margin: auto; }
    .header { text-align: center; padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 10px; margin-bottom: 20px; }
    .header h1 { color: white; margin: 0; }
    .header p { color: #e0e0e0; margin: 10px 0 0 0; }
    """
) as demo:
    
    gr.HTML("""
    <div class="header">
        <h1>TechCorp HR Assistant</h1>
        <p>Ask me about policies, benefits, leave, and more!</p>
    </div>
    """)
    
    chatbot = gr.Chatbot(height=400, bubble_full_width=False)
    
    with gr.Row():
        msg = gr.Textbox(
            placeholder="Ask an HR question...",
            show_label=False,
            scale=9
        )
        submit = gr.Button("Send", variant="primary", scale=1)
    
    clear = gr.Button("Clear Chat")
    
    gr.Examples(
        examples=[
            "How many vacation days do I get?",
            "What is the remote work policy?",
            "How does performance review work?",
            "What are the health benefits?",
            "How much maternity leave is provided?",
            "What are the password requirements?",
        ],
        inputs=msg,
        label="Example Questions"
    )
    
    def respond(message, history):
        if not message.strip():
            return history, ""
        response = hr_bot.chat(message)
        history.append([message, response])
        return history, ""
    
    def clear_chat():
        hr_bot.clear_history()
        return [], ""
    
    msg.submit(respond, [msg, chatbot], [chatbot, msg])
    submit.click(respond, [msg, chatbot], [chatbot, msg])
    clear.click(clear_chat, outputs=[chatbot, msg])

demo.launch()

---
## 11. Cleanup

In [ ]:
import shutil

cleanup = False  # Set to True to remove files

if cleanup:
    for path in ["hr_chroma_db", "hr_documents"]:
        if os.path.exists(path):
            shutil.rmtree(path)
            print(f"Removed {path}")
    print("Cleanup complete!")
else:
    print("Cleanup skipped. Set cleanup=True to remove files.")

---
## Summary

| Topic | Technique |
|-------|----------|
| **Document Loading** | PyPDFLoader, CSVLoader, TextLoader, DirectoryLoader |
| **Chunking** | RecursiveCharacterTextSplitter (1000 chars, 200 overlap) |
| **Embeddings** | HuggingFaceEmbeddings (all-MiniLM-L6-v2) |
| **Vector Store** | ChromaDB |
| **Basic Retrieval** | Similarity search |
| **Advanced Retrieval** | MMR, Multi-Query, Compression, Threshold |
| **Conversational RAG** | History-aware retriever |
| **UI** | Gradio Blocks |

### Key Takeaways
1. **Chunk wisely**: 500-1000 chars with 10-20% overlap
2. **Use MMR** for diverse results
3. **Add chat history** for follow-up questions
4. **Always cite sources** in responses

---
*Advanced RAG Workshop with LangChain*